# 02 · Fine-tune Qwen2.5-Coder (QLoRA) for Text2SQL — Colab T4

Free-tier recipe: **Unsloth + QLoRA** on `Qwen2.5-Coder-1.5B-Instruct`
(Apache-2.0, 1.54B params, code-specialised). ~1.5–2 GB VRAM for the adapter;
fits a free T4 with room to spare.

> Runtime → Change runtime type → **T4 GPU** before running.

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 1. Install

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

## 2. Get the code + data
Clone your repo (or upload `src/`). Then download BIRD.

In [ ]:
# !git clone https://github.com/<you>/text2sql-finetuning.git && cd text2sql-finetuning
import sys; sys.path.insert(0, '.')  # ensure `src` importable

In [ ]:
# --- BIRD (official). Skip if you uploaded your own processed JSONL. ---
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/train.zip && unzip -q -o train.zip
!wget -q https://bird-bench.oss-cn-beijing.aliyuncs.com/dev.zip   && unzip -q -o dev.zip
# BIRD nests databases one level deep; adjust paths if the archive layout differs.

## 3. Preprocess → JSONL
Builds prompt/messages records with the schema reconstructed from each .sqlite.

In [ ]:
!python -m src.data_prep --source bird --json train/train.json \
    --db_root train/train_databases --out data/processed/bird_train.jsonl --shuffle
!python -m src.data_prep --source bird --json dev/dev.json \
    --db_root dev/dev_databases --out data/processed/bird_dev.jsonl

## 4. Train (QLoRA)
Main experiment (`exp1`). For a fast sanity run add `--max_steps 30`.
Subsample with `--max_train_samples 5000` if you're time-boxed on Colab.

In [ ]:
!python -m src.train --preset exp1_qwen1.5b_bird_qlora \
    --train_file data/processed/bird_train.jsonl \
    --val_file   data/processed/bird_dev.jsonl \
    --max_train_samples 8000 --epochs 2

### (Optional) inline training loop
Equivalent to the CLI above, exposed here so you can tweak interactively.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

model, tok = FastLanguageModel.from_pretrained(
    'unsloth/Qwen2.5-Coder-1.5B-Instruct', max_seq_length=2048, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth', random_state=3407)

ds = load_dataset('json', data_files='data/processed/bird_train.jsonl', split='train')
ds = ds.map(lambda e: {'text': [tok.apply_chat_template(m, tokenize=False) for m in e['messages']]},
            batched=True, remove_columns=ds.column_names)

trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(output_dir='outputs/exp1', per_device_train_batch_size=8,
        gradient_accumulation_steps=2, num_train_epochs=2, learning_rate=2e-4,
        lr_scheduler_type='cosine', warmup_ratio=0.03, optim='adamw_8bit',
        logging_steps=20, bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported(),
        dataset_text_field='text', max_seq_length=2048, report_to='none'))
trainer = train_on_responses_only(trainer,
    instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
model.save_pretrained('outputs/exp1'); tok.save_pretrained('outputs/exp1')

## 5. (Optional) push the adapter to the Hub

In [ ]:
# from huggingface_hub import login; login()
# model.push_to_hub('<you>/qwen2.5-coder-1.5b-bird-qlora')
# tok.push_to_hub('<you>/qwen2.5-coder-1.5b-bird-qlora')

Proceed to **03_inference_eval.ipynb** to measure execution accuracy.